# Map of phone clips

Turns one or more phone videos of a place into one 3D map of its objects and shows it in Rerun: the room as a point cloud, the camera's path,
and every object as a labelled box. The steps:

1. **Send the clip** from the phone: Share → Tailscale → beast. The first cell collects it into `data/phone/`.
2. **Set `CLIPS` and `PROMPTS`** in the settings cell: which videos, and what objects SAM 3 should look for.
   Several videos of the same place become one map: VGGT-Omega puts them in one 3D frame and the graph links
   the objects they share.
3. **Run all cells.** The pipeline step reuses what is already computed, so a clip you have run before shows up at
   once. New clips, or new prompts, take a minute or two.

Use the `EAI project` kernel (see the README's Setup). The pipeline is `run.py`; this notebook only calls it
and shows the result.

In [ ]:
import math
import subprocess
import sys

from torchcodec.decoders import VideoDecoder

from models import vggt
from utils import footage
from utils.config import ROOT_DIR

# Clips sent with Tailscale wait in its inbox until collected
subprocess.run(["tailscale", "file", "get", "--conflict=rename", str(footage.PHONE_DIR)], check=False)

print("Clips in data/phone:")
for path in sorted(footage.PHONE_DIR.iterdir()):
    meta = VideoDecoder(str(path)).metadata
    print(f"  {path.stem:<20} {meta.duration_seconds:5.1f} s, {meta.num_frames} frames at {meta.average_fps:.0f} fps")

In [ ]:
CLIPS = ["IMG_0649",]   # videos in data/phone/ of one place, without their extensions
PROMPTS = ["umbrella", "monitor", "table", "chair", "coffee"]  # what SAM 3 looks for; None for 60 indoor objects
STRIDE = None  # use every STRIDE-th frame; None picks about 10 frames per second of video

name = "+".join(CLIPS)
metas = [VideoDecoder(str(p)).metadata for p in footage.video_paths(footage.make_run(f"phone/{name}", 1, "sam3", None))]
total = sum(m.num_frames for m in metas)
stride = STRIDE or max(round(metas[0].average_fps / 10), math.ceil(total / vggt.MAX_FRAMES), 1)
args = ["--source", f"phone/{name}", "--stride", str(stride)]
if PROMPTS:
    vocab_file = ROOT_DIR / "data" / "vocab" / f"{name}.txt"
    vocab_file.parent.mkdir(exist_ok=True)
    vocab_file.write_text("\n".join(PROMPTS) + "\n")
    args += ["--vocab", str(vocab_file)]
run = footage.make_run(f"phone/{name}", stride, "sam3", None, name if PROMPTS else "")
print(f"{name}: stride {stride}, {sum(math.ceil(m.num_frames / stride) for m in metas)} frames")


def sh(*command: str) -> None:
    """Run a command in the repository and show its output here, without warnings and progress bars."""
    proc = subprocess.Popen(
        [sys.executable, *command], cwd=ROOT_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    noise = ("Warning", "warnings.warn", "[transformers]", "Loading weights", "it/s]", "s/it]", "from_numpy")
    for line in proc.stdout:
        if line.strip() and not any(n in line for n in noise):
            print(line, end="")
    if proc.wait():
        raise RuntimeError(f"{command[0]} failed")

## Run the pipeline

In [ ]:
sh("run.py", *args)

## The map in Rerun

The room is frozen: the point cloud fuses the depth of 40 frames from VGGT-Omega, in metres with z up. The lines are
the camera's paths, one colour per video (hover for its name). Scrub the `frame` timeline at the bottom to move the camera along it; the right view shows what
it saw, with the objects' 2D boxes. Hover a box for its label; colours match between the views and the video below.
The heading follows the first frame, not the walls, so objects on walls at an angle get roomier boxes.

In [ ]:
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)  # torch.jit, via transformers

import rerun as rr

from utils import map_view

rr.init("map")
rr.notebook_show(height=750)
map_view.log_map(run)

## The graph: which tracks became one object, and why

SAM 3 gives tracks; the graph merges them into objects (`graph/graph.py`). On top, every track over the run's frames,
grouped and coloured by object: tracks of one object that overlap in time are duplicates, ones that follow each other
are re-entries, and ones on both sides of the dashed line link the two videos. Below, the objects from above with the
same numbers and colours as in Rerun, and on the right every merge with its rule and value, and every near-miss with
what kept the two apart. `python -m graph.plot` saves the same picture to `eval/graph.png`.

In [ ]:
from graph import plot

plot.plot(run);

## The video with every object outlined and labelled

Several clips play one after the other; the header names the clip.

In [ ]:
from IPython.display import Video

sh("-m", "utils.video", *args, "--panels", "graph", "--fps", str(round(metas[0].average_fps / stride, 2)))
Video(str(run.stage_dir("eval") / "graph.mp4"), embed=True, width=420)

## Search the map by text

In [ ]:
sh("-m", "graph.query", *args, "umbrella")